# 02b — A3 con presupuesto extendido (CISUTAC Holes)

Este notebook ejecuta **únicamente** la variante ganadora de validación del notebook 02:

- YOLO11n preentrenado en COCO.
- Tiles de 640 × 640.
- Solapamiento del 20 %.
- Un tile sano por cada tile con defecto (`NEG_POR_POS = 1`).
- Presupuesto extendido: hasta 100 épocas, con paciencia 30.

La A3 original fue limitada a 26 épocas para igualar imágenes vistas con A0 y obtuvo **mAP@0.5 = 0,131** en validación. Como su mejor época fue la última, esta corrida comprueba si A3 seguía mejorando.

> **Este notebook no evalúa test.** Sólo usa `valid`. El test debe abrirse una única vez después de comparar esta corrida con la A3 original y congelar la variante definitiva.

## 0. Preparación para Kaggle

Activar una GPU e Internet. La celda instala dependencias, clona el repositorio y descarga CISUTAC Holes si todavía no está disponible.

In [ ]:
%pip install -q ultralytics==8.4.215 sahi==0.14.0 pycocotools==2.0.11 huggingface_hub

import os
import subprocess
import sys
from pathlib import Path

ROOT = Path('/kaggle/working/tp_final_vision')
DATA_ROOT = Path('/kaggle/working/rewear_data')

if not ROOT.exists():
    subprocess.run([
        'git', 'clone', '-q',
        'https://github.com/victoriapersico/tp_final_vision.git',
        str(ROOT),
    ], check=True)

os.environ['REWEAR_DATA_DIR'] = str(DATA_ROOT)
os.chdir(ROOT)
sys.path.insert(0, str(ROOT / 'src'))

holes_dir = DATA_ROOT / 'cisutac_holes'
spots_dir = DATA_ROOT / 'cisutac_spots'
if not (holes_dir / 'train' / 'images').exists() or not (spots_dir / 'train' / 'images').exists():
    # El preparador oficial genera ambos datasets en una sola pasada.
    subprocess.run([sys.executable, str(ROOT / 'scripts' / 'download_datasets.py'), 'all'], check=True)

print('Repositorio:', ROOT)
print('Datos:', holes_dir)

## 1. Parámetros congelados

El único cambio respecto de la A3 original es el presupuesto de épocas. El nombre nuevo evita sobrescribir la corrida de Pepo.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

import cisutac_baseline as cb

DATASET = 'holes'
MODELO = 'yolo11n.pt'
SEMILLA = 0
TILE = 640
SOLAPAMIENTO = 0.20
NEG_POR_POS = 1.0
MIN_VISIBLE = 0.50
EPOCAS_EXTENDIDAS = 100
PACIENCIA = 30
BATCH = 16
CONF_EVAL = 0.001
NOMBRE_CORRIDA = 'holes_A3_tiles640_neg1_ext100'
SPLIT_EVALUACION = 'valid'

assert SPLIT_EVALUACION == 'valid', 'Este notebook no debe abrir test.'
print('Dispositivo:', cb.pick_device())
print('Resultados:', cb.RESULTS)

## 2. Preparación de datos y tiles

Se conserva exactamente el split y la construcción de tiles del notebook 02.

In [ ]:
if not (cb.derived_dir(DATASET) / 'data.yaml').exists():
    subprocess.run([sys.executable, str(ROOT / 'scripts' / 'prepare_cisutac_yolo.py')], check=True)

yaml_tiles, stats_tiles = cb.build_tile_dataset(
    DATASET,
    tile=TILE,
    overlap=SOLAPAMIENTO,
    neg_per_pos=NEG_POR_POS,
    seed=SEMILLA,
    min_visible=MIN_VISIBLE,
)

display(pd.DataFrame(stats_tiles).T)
print('Dataset YOLO de tiles:', yaml_tiles)

## 3. Entrenamiento A3 extendido

A diferencia de la A3 original, esta corrida permite hasta 100 épocas. El *early stopping* puede detenerla antes si validación deja de mejorar durante 30 épocas.

In [ ]:
run_ext = cb.train(
    NOMBRE_CORRIDA,
    yaml_tiles,
    imgsz=TILE,
    epochs=EPOCAS_EXTENDIDAS,
    patience=PACIENCIA,
    batch=BATCH,
    seed=SEMILLA,
    model=MODELO,
    retrain=False,
)

cb.plot_curves({'A3 extendida': run_ext});

## 4. Evaluación únicamente en validación

In [ ]:
eval_ext = cb.evaluate(
    f'{NOMBRE_CORRIDA}__valid_tiles640_o20',
    DATASET,
    Path(run_ext) / 'weights' / 'best.pt',
    split=SPLIT_EVALUACION,
    mode='tiles',
    tile=TILE,
    overlap=SOLAPAMIENTO,
    also_full=False,
    conf=CONF_EVAL,
    reevaluate=False,
)

display(cb.metrics_table([eval_ext]))
display(cb.size_table([eval_ext]))
cb.plot_recall_by_size([eval_ext]);

## 5. Comparación con la A3 original

La fila original usa los resultados guardados por Pepo. La decisión se toma por mAP@0.5 en `valid`; test continúa cerrado.

In [ ]:
comparacion = pd.DataFrame([
    {
        'variante': 'A3 original — presupuesto igualado (26 épocas máx.)',
        'mAP@0.5': 0.131,
        'mAP@0.5:0.95': 0.039,
        'AP_small': 0.077,
        'recall': 0.225,
        'segundos_por_imagen': 0.454,
    },
    {
        'variante': 'A3 extendida — hasta 100 épocas',
        'mAP@0.5': eval_ext['mAP@0.5'],
        'mAP@0.5:0.95': eval_ext['mAP@0.5:0.95'],
        'AP_small': eval_ext['AP_small'],
        'recall': eval_ext['recall'],
        'segundos_por_imagen': eval_ext['segundos_por_imagen'],
    },
]).set_index('variante')
display(comparacion)

if eval_ext['mAP@0.5'] > 0.131:
    print('La A3 extendida supera el mAP@0.5 de validación de la original.')
elif eval_ext['mAP@0.5'] == 0.131:
    print('Empate en mAP@0.5: se prefiere la corrida de menor presupuesto.')
else:
    print('La A3 original conserva el mejor mAP@0.5 de validación.')

## 6. Próximo paso

1. Guardar esta corrida con **Save Version → Save & Run All**.
2. Confirmar que finalice como `Successful`.
3. Comparar el mAP@0.5 de validación con 0,131.
4. Congelar la variante elegida.
5. Recién después preparar la evaluación única en test en otro notebook o versión explícita.

No modificar hiperparámetros usando resultados de test.